# 03 - Statistical analysis and predictive evaluation

Everything here is read back from the saved artifacts of a completed run:

```bash
python -m eeg_research_agent.cli run --mode smoke
```

The point of the notebook is the *audit*: the primary analysis is at the subject
level, effect sizes accompany every p-value, p-values are FDR-adjusted, and any
claim in the reports can be re-read from an artifact.

In [ ]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
tables = ROOT / 'artifacts/tables'
stats = json.loads((tables / 'statistical_results.json').read_text())
print('unit of analysis :', stats['unit_of_analysis'])
print('subjects / epochs:', stats['n_subjects'], '/', stats['n_epochs'])
print('features tested  :', stats['n_features'])
print('correction       :', stats['family_correction'])
print('significant after FDR:', stats['family_correction']['n_rejected'])
raw_significant = sum(1 for c in stats['comparisons']
                      if (c.get('primary') or {}).get('p_value', 1) <= stats['alpha'])
print('significant before FDR:', raw_significant)

## Per-feature comparisons

Every row carries the selected test, the assumption checks that drove the choice,
an effect size and both raw and adjusted p-values.

In [ ]:
rows = []
for comparison in stats['comparisons']:
    primary = comparison.get('primary') or {}
    effect = primary.get('effect_size') or {}
    rows.append({
        'feature': comparison['feature'],
        'stages': len(comparison.get('stages') or []),
        'unit': comparison.get('unit_of_analysis'),
        'test': primary.get('test'),
        'p_raw': primary.get('p_value'),
        'p_fdr': primary.get('p_adjusted'),
        'effect': effect.get('measure'),
        'effect_value': effect.get('value'),
        'significant': primary.get('significant_after_correction'),
    })
comparisons = pd.DataFrame(rows)
print('skipped comparisons:', int(comparisons['test'].isna().sum()))
print(comparisons['test'].value_counts().to_string())

In [ ]:
top = comparisons.dropna(subset=['p_fdr']).sort_values('p_fdr').head(15)
display(top[['feature', 'test', 'p_raw', 'p_fdr', 'effect', 'effect_value']].round(4))

## Predictive evaluation

Held-out metrics on **unseen subjects**. Balanced accuracy and macro F1 are the
metrics that matter on imbalanced sleep data; N1/N2/REM are broken out because
they are the difficult distinctions.

In [ ]:
evaluation = json.loads((tables / 'predictive_evaluation.json').read_text())
test = pd.DataFrame([r for r in evaluation['results'] if r.get('split') == 'test'])
columns = ['feature_set', 'classifier', 'n_features', 'balanced_accuracy',
           'macro_f1', 'cohens_kappa']
best = test.sort_values('balanced_accuracy', ascending=False).head(10)
display(best[columns].round(3))

In [ ]:
focused = test[test['feature_set'].isin(['all_features', 'conventional_only', 'novel_only'])]
pivot = focused.pivot_table(index='feature_set', columns='classifier',
                            values='balanced_accuracy', aggfunc='max')
display(pivot.round(3))

## The evidence trail

Every number printed in `reports/*.md` is registered here with the artifact it
came from. `validate_claims` re-reads those artifacts; a report that drifts from
its data fails the run.

In [ ]:
claims = json.loads((tables / 'claims.json').read_text())
print('claims registered:', claims['n_claims'])
claim_frame = pd.DataFrame(claims['claims'])
display(claim_frame[['claim_id', 'level', 'value', 'unit', 'source']].head(15))

In [ ]:
from eeg_research_agent.verification.research_validation import validate_claims

result = validate_claims(claims['claims'], {}, artifacts_dir=ROOT / 'artifacts')
print('verification ok :', result['ok'])
print('n verified      :', result['n_verified'])
print('failures        :', result['failures'])